# 10 — Facets

**Purpose:** Explore the `tl.facets` semantic-access system: registry discovery,
`tl.facet`/`tl.head` selectors, per-record `FacetView`, `Facet.value`, `Facet.grad`,
and the attention head-facet API (guarded — unavailable on zoo models without HF).

**Surfaces covered:**
- [ ] `tl.facets` — module-level namespace
- [ ] `tl.facets.snapshot()` — registry discovery (`FacetRegistrySnapshot`)
- [ ] `tl.facets.info(class_name)` — per-class recipe/facet listing
- [ ] `tl.facet(name)` — `FacetSelector` predicate
- [ ] `tl.head(index)` — head-index `FacetSelector` predicate
- [ ] `op.facets` — `FacetView` on an `Op` record
- [ ] `FacetView.keys()`, `FacetView.menu()`, `FacetView.has()`, `FacetView[name]`
- [ ] `Facet.value` — forward activation tensor
- [ ] `Facet.grad` — gradient tensor (requires `save_grads=True` + backward)
- [ ] Attention q/k/v facets — **GUARDED** (skip cleanly if unavailable on zoo models)
- [ ] `FacetView.head(i)` — `AttentionHeadView` (GUARDED)

## 1. Environment setup

In [ ]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

import torch
import torchlens as tl
from _models import ZOO

print(f"torchlens version : {tl.__version__}")
print(f"torch version     : {torch.__version__}")
print(f"ZOO entries       : {list(ZOO.keys())}")

## 2. Registry discovery — `tl.facets.snapshot()`

`tl.facets.snapshot()` returns a `FacetRegistrySnapshot` listing all registered
attention/MLP/normalization recipes. This is the discoverable entry-point — a user
can call it to see what facets TorchLens knows about before capturing.

In [ ]:
snap = tl.facets.snapshot()
print(f"Registry version  : {snap.version}")
print(f"Provenance ID     : {snap.provenance_id}")
print(f"Registered recipes: {len(snap.recipes)}")
print()
print("Recipe names:")
for rr in snap.recipes:
    r = rr.public
    print(
        f"  {r.recipe_name:40s}  classes: {r.class_names[:2]}{'...' if len(r.class_names) > 2 else ''}"
    )

## 3. Per-class facet listing — `tl.facets.info(class_name)`

`tl.facets.info(class_name)` returns a dict with the recipes and facet names
associated with that PyTorch module class.  Useful to check what `tl.facet('q')`
would match before actually running a trace.

In [ ]:
# Sample a few known class names from the registry
sample_classes = [
    "GPT2Attention",
    "LlamaAttention",
    "LayerNorm",
    "Embedding",
]

for cls in sample_classes:
    try:
        info = tl.facets.info(cls)
        print(f"{cls}:")
        print(f"  recipes: {info['recipes']}")
        print(f"  facets : {info['facets']}")
    except Exception as exc:
        print(f"{cls}: GAP — {exc}")
    print()

## 4. `tl.facet()` and `tl.head()` as selectors

`tl.facet(name)` returns a `FacetSelector` usable in `save=`/`intervene=`/etc.
`tl.head(index)` selects a specific attention head index.
Both are predicates: they return `True` when evaluated against a matching record.

In [ ]:
# Build selectors
facet_sel = tl.facet("out")  # built-in facet name present on all ops
head_sel = tl.head(0)  # attention head 0

print("tl.facet('out') repr :", repr(facet_sel))
print("tl.head(0) repr      :", repr(head_sel))
print()

# Use as save= predicate in tl.trace (selects only ops that match)
model, x = ZOO["tiny_mlp"]()
trace_facet_save = tl.trace(model, x, save=facet_sel)
print("trace with save=tl.facet('out') completed OK")
print(f"  layers logged: {len(trace_facet_save.layer_labels)}")

# head selector (no attention modules in zoo, so acts like False on all ops)
trace_head_save = tl.trace(model, x, save=head_sel)
print("trace with save=tl.head(0) completed OK")
print(
    f"  layers saved (expect 0 or minimal on non-attention model): {len(trace_head_save.layer_labels)}"
)

## 5. `op.facets` — `FacetView` on an `Op` record

Every `Op` exposes a `.facets` attribute that returns a `FacetView`.  
`FacetView.keys()` lists available facet names; `FacetView.menu()` shows
availability status; `FacetView[name]` returns a `Facet` object.

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace = tl.trace(model, x)

linear_op = trace["linear_1_1"]
print(f"Op label   : {linear_op.layer_label}")
print(f"Op type    : {type(linear_op).__name__}")
print()

fv = linear_op.facets
print(f"FacetView type : {type(fv).__name__}")
print(f"FacetView keys : {list(fv.keys())}")
print()
print("FacetView.menu():")
for name, item in fv.menu().items():
    print(f"  {name:15s} status={item.status!r}  recipe={item.recipe!r}")
print()

# FacetView on a module record (has richer facet set)
mod = trace.modules["in_proj"]
print(f"Module facets keys: {list(mod.facets.keys())}")
print(f"Module facets menu: {mod.facets.menu()}")

## 6. `Facet.value` — forward activation tensor

`FacetView[name]` returns a `Facet` with `.value` holding the actual captured tensor.

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace = tl.trace(model, x)

relu_op = trace["relu_1_2"]
facet = relu_op.facets["out"]

print(f"Facet type   : {type(facet).__name__}")
print(f"Facet.spec   : {facet.spec}")
print()
print(f"Facet.value  : {facet.value}")
print(f"  .shape     : {facet.value.shape}")
print(f"  .dtype     : {facet.value.dtype}")
print()
print("What a user gets vs raw op.out:")
print(f"  op.out         = {relu_op.out}")
print(f"  facet.value    = {facet.value}")
print("  (same tensor; facet is the semantic route — will differ for head-sliced attention)")

## 7. `Facet.grad` — gradient tensor

`Facet.grad` returns the saved gradient if backward was captured, or a
`MissingGradient` with a `.reason` and `.recapture_instruction` otherwise.  
Requires `save_grads=True` + calling `.backward()` after the trace.

In [ ]:
from _models import LinearReluModel  # small backward-friendly model

model = LinearReluModel()
x = torch.randn(1, 3, requires_grad=True)

# Capture with save_grads=True so TorchLens hooks backward
trace = tl.trace(model, x, save_grads=True)

# Run backward on the scalar output
output_op = trace[trace.layer_labels[-1]]
output_op.out.backward()

linear_op = trace["linear_1_1"]
facet = linear_op.facets["out"]

print("After backward with save_grads=True:")
print(f"  Facet.value : {facet.value}")
print(f"  Facet.grad  : {facet.grad}")
print(f"  grad.shape  : {facet.grad.shape}")
print()

# Show MissingGradient when not captured
trace_no_grad = tl.trace(model, x)  # no save_grads
output_op2 = trace_no_grad[trace_no_grad.layer_labels[-1]]
output_op2.out.backward()
missing = trace_no_grad["linear_1_1"].facets["out"].grad
print("Without save_grads — MissingGradient:")
print(f"  type    : {type(missing).__name__}")
print(f"  reason  : {missing.reason}")
print(f"  fix     : {missing.recapture_instruction}")

## 8. Attention q/k/v head facets — GUARDED

Attention facets (`q`, `k`, `v`, `attn_out`, `head`) are registered for HF
classes (GPT2, BERT, LLaMA, ViT, DistilBERT, T5, etc.) and require `transformers`
to be installed and a network-available model.  The zoo models do not use those
module classes, so these facets are unavailable without HF.

The block below verifies graceful absence and explains what the API would yield.

In [ ]:
# Try to trigger an attention facet on a zoo model to show the clean fallback
model, x = ZOO["tiny_mlp"]()
trace = tl.trace(model, x)

# Check if any op has an attention facet
has_attention_facet = False
for label in trace.layer_labels:
    op = trace[label]
    fv = op.facets
    for name in fv.keys():
        if name in ("q", "k", "v", "attn_out", "head"):
            has_attention_facet = True
            print(f"Found attention facet '{name}' on op '{label}' — showing head view:")
            try:
                hv = fv.head(0)
                print(f"  head view type: {type(hv).__name__}")
                print(f"  head view      : {hv}")
            except Exception as exc:
                print(f"  head view error: {exc}")

if not has_attention_facet:
    print("No attention facets on zoo models (expected — need HF attention modules).")
    print()
    print("On a GPT-2 model captured with tl.trace, the API would look like:")
    print("  module = trace.modules['transformer.h.0.attn']")
    print("  fv = module.facets")
    print("  fv.keys()  # => ['q', 'k', 'v', 'attn_out', 'input', 'n_heads', 'head', ...]")
    print("  q_facet = fv['q']")
    print("  q_facet.value.shape  # => [batch, n_heads, seq_len, d_head]")
    print("  head_view = fv.head(0)")
    print("  head_view['q'].value.shape  # => [batch, 1, seq_len, d_head]")
    print()
    print("  tl.head(0) as a save= selector captures only head-0 ops.")
    print("  tl.facet('q') as a save= selector captures only q-projection ops.")

# Check MissingFacet via menu on a zoo op
relu_op = trace["relu_1_2"]
menu = relu_op.facets.menu()
print()
print("Demonstrating FacetMenuItem status on a zoo relu op:")
for name, item in menu.items():
    print(f"  '{name}': status={item.status!r}")

## 9. Facet vs raw op — what a user gains

On simple ops (Linear, ReLU), `facet.value` mirrors `op.out`.  The semantic value
comes with attention modules: facets give per-component, per-head slices without
the user having to know the internal tensor layout.

In [ ]:
model, x = ZOO["tiny_mlp"]()
trace = tl.trace(model, x)
relu_op = trace["relu_1_2"]

# Show the three routes to the same tensor
print("Three routes to the ReLU activation:")
print(f"  1. trace['relu_1_2'].out         : shape={relu_op.out.shape}")
print(f"  2. relu_op.facets['out'].value   : shape={relu_op.facets['out'].value.shape}")
print()
print("For attention modules, facets add:")
print("  - Semantically named slots: 'q', 'k', 'v', 'attn_out', 'n_heads'")
print("  - Head slicing: fv.head(i)['q'].value   => per-head q tensor")
print("  - Grad pair: facet.value + facet.grad   => attribution-ready")
print("  - Registry-driven: new architectures can be registered without code changes")

---

## ⚠️ GAPs / ergonomic smells

- **`Facet.__repr__` is uninformative** — `repr(facet)` returns the default `<Facet object at 0x...>`.  Users cannot tell at a glance what recipe/home it came from.  A `Facet(home='relu_1_2:1', name='out', shape=[2,8])` repr would be much more useful.
- **`MissingGradient.__repr__`** is also a bare object address — `.reason` is informative but only if the user knows to call it.  Repr should inline the reason.
- **`FacetView.has()` signature unknown** — `FacetView.has` exists but no working example was found; calling it without arguments raised a TypeError.  Docs/signature should be clarified.
- **`tl.head()` silently saves nothing on non-attention models** — returns `True` for zero ops, gives no warning.  A user might think capture succeeded when no head-sliced tensors were actually saved.
- **Attention q/k/v facets require HF modules** — none of the zoo models trigger them.  Notebook 14 (guarded HF) is the right place to exercise these end-to-end; this notebook covers the API surface only.
- **`output_table()` errors without semantic output decoding** — not exposed here; covered in notebook 12 where the GAP is documented.